# Algorithm 8 — Order-Statistic Percentile

**File:** `src/CopilotScope.Collector/Domain/CopilotSession.cs` (line 193)

A **ceil-based** order-statistic used throughout the analyzer pipeline for p50/p95 TTFT
computation and the session composite score.

---

## 1  Definition

Given a sorted list $v_{(1)} \leq v_{(2)} \leq \cdots \leq v_{(n)}$ and quantile $p \in (0, 1]$:

$$
\boxed{\text{Percentile}(V, p) = v_{\left(\,\text{clamp}\!\left(\lceil p \cdot n \rceil - 1,\; 0,\; n-1\right)\right)}}
$$

This is equivalent to the **nearest-rank** method (also called the "ceiling" or "exclusive" method),
and differs from linear interpolation (used by NumPy's default `np.percentile`).

---

## 2  Edge Cases

| Input | Result |
|---|---|
| $n = 0$ | 0 (empty guard) |
| $p = 0$ | $v_{(1)}$ (clamped to index 0) |
| $p = 1$ | $v_{(n)}$ (maximum) |
| $n = 1$ | $v_{(1)}$ for all $p$ |

---

## 3  Comparison with Linear Interpolation

For small $n$, the ceil method and linear interpolation can diverge meaningfully.  The ceil
method always returns an **actual observed value** — which is desirable for latency reporting
(a real TTFT, not a hypothetical interpolated one).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

def percentile_ceil(values: list[float], p: float) -> float:
    """Exact replica of CopilotSession.Percentile (ceil / nearest-rank)."""
    if not values: return 0.0
    sorted_v = sorted(values)
    n = len(sorted_v)
    idx = int(np.clip(math.ceil(p * n) - 1, 0, n - 1))
    return sorted_v[idx]

def percentile_linear(values: list[float], p: float) -> float:
    """NumPy linear interpolation (for comparison)."""
    if not values: return 0.0
    return float(np.percentile(values, p * 100))

# --- Correctness verification ---
print('=== Correctness: Percentile(V, p) ===')
v = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
print(f'V = {v}')
print(f'{"p":>5}  {"ceil (ours)":>12}  {"linear (numpy)":>15}')
for p in [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0]:
    c = percentile_ceil(v, p)
    l = percentile_linear(v, p)
    print(f'{p:>5.2f}  {c:>12.1f}  {l:>15.1f}')

print('\n=== Edge cases ===')
print(f'Empty list:          {percentile_ceil([], 0.5)}')
print(f'Single element p50:  {percentile_ceil([42.0], 0.5)}')
print(f'Single element p95:  {percentile_ceil([42.0], 0.95)}')
print(f'Two elements p50:    {percentile_ceil([10.0, 20.0], 0.5)}')
print(f'Two elements p95:    {percentile_ceil([10.0, 20.0], 0.95)}')

In [ ]:
# Divergence between ceil and linear at small n
rng = np.random.default_rng(7)
sample_sizes = [3, 5, 10, 20, 50, 100]
n_trials = 500
p = 0.95
divergences = []
for n in sample_sizes:
    diffs = []
    for _ in range(n_trials):
        v = list(rng.lognormal(6.5, 0.5, n))
        diffs.append(abs(percentile_ceil(v, p) - percentile_linear(v, p)))
    divergences.append(np.mean(diffs))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sample_sizes, divergences, 'o-', color='steelblue', lw=2)
ax.set_xlabel('Sample size n')
ax.set_ylabel('Mean |ceil − linear| at p95 (ms)')
ax.set_title('Ceil vs. linear interpolation divergence (lognormal TTFT, p95)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('percentile_divergence.png', dpi=150)
plt.show()